In [1]:
# Optimal Power Flow Solution Using Cuckoo Search Algorithm

In [5]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.special as special

# Set random seed for reproducibility
np.random.seed(42)

# Generator Cost Coefficients (a * P^2 + b * P + c)
# Let's assume a 3-generator system
gen_costs = [
    {'a': 0.008, 'b': 2.0, 'c': 100, 'Pmin': 10, 'Pmax': 85},
    {'a': 0.009, 'b': 1.8, 'c': 120, 'Pmin': 10, 'Pmax': 80},
    {'a': 0.007, 'b': 2.1, 'c': 80,  'Pmin': 10, 'Pmax': 70}
]

# System demand (MW)
PD = 150.0

# Transmission loss coefficients (B-coefficients matrix for simplifying line losses)
B_matrix = np.array([
    [0.00003, 0.00001, 0.000013],
    [0.00001, 0.00009, 0.00001],
    [0.000013, 0.00001, 0.00012]
])

def calculate_loss(P):
    """Calculate transmission loss using B-matrix."""
    return np.dot(P, np.dot(B_matrix, P))

def objective_function(P):
    """Calculate total fuel cost + penalty for violating power balance constraint."""
    total_cost = 0
    for i, gen in enumerate(gen_costs):
        total_cost += gen['a'] * (P[i]**2) + gen['b'] * P[i] + gen['c']

    # Transmission loss
    Ploss = calculate_loss(P)

    # Equality Constraint: Sum(P) - Ploss - PD = 0
    power_balance_violation = abs(np.sum(P) - Ploss - PD)

    # Penalty factor for constraint violation
    penalty = 100000.0 * (power_balance_violation ** 2)

    return total_cost + penalty


In [6]:
def levy_flight(Lambda, size):
    """Generate step sizes using Levy Flight distribution."""
    # Calculate sigma using gamma functions
    num = special.gamma(1 + Lambda) * np.sin(np.pi * Lambda / 2)
    den = special.gamma((1 + Lambda) / 2) * Lambda * (2 ** ((Lambda - 1) / 2))
    sigma = (num / den) ** (1 / Lambda)

    u = np.random.normal(0, sigma, size=size)
    v = np.random.normal(0, 1, size=size)
    step = u / (np.abs(v) ** (1 / Lambda))
    return step

def cuckoo_search_opf(n_nests=25, max_iter=200, pa=0.25):
    n_dim = len(gen_costs)
    # Get boundaries
    p_min = np.array([g['Pmin'] for g in gen_costs])
    p_max = np.array([g['Pmax'] for g in gen_costs])

    # Initialize nest positions
    nests = np.random.uniform(p_min, p_max, (n_nests, n_dim))

    # Evaluate fitness
    fitness = np.array([objective_function(nest) for nest in nests])

    # Find the current best nest
    best_idx = np.argmin(fitness)
    best_nest = nests[best_idx].copy()
    best_fitness = fitness[best_idx]

    history = []

    for t in range(max_iter):
        # 1. Generate new solutions via Levy flight from a random nest
        new_nests = nests.copy()
        for i in range(n_nests):
            step_size = levy_flight(1.5, n_dim)
            # Random walk starting from the current best
            new_nests[i] += step_size * (nests[i] - best_nest)
            # Enforce boundary limits
            new_nests[i] = np.clip(new_nests[i], p_min, p_max)

        # Evaluate fitness of new nests
        new_fitness = np.array([objective_function(nest) for nest in new_nests])

        # Greedy replacement selection
        for i in range(n_nests):
            if new_fitness[i] < fitness[i]:
                nests[i] = new_nests[i]
                fitness[i] = new_fitness[i]

        # 2. Abandon a fraction (pa) of worst nests and build new ones
        for i in range(n_nests):
            if np.random.rand() < pa:
                # Discover and change via random step
                random_nest_1 = nests[np.random.randint(0, n_nests)]
                random_nest_2 = nests[np.random.randint(0, n_nests)]
                step_factor = np.random.rand()
                nests[i] += step_factor * (random_nest_1 - random_nest_2)
                nests[i] = np.clip(nests[i], p_min, p_max)
                fitness[i] = objective_function(nests[i])

        # Update global best
        current_best_idx = np.argmin(fitness)
        if fitness[current_best_idx] < best_fitness:
            best_fitness = fitness[current_best_idx]
            best_nest = nests[current_best_idx].copy()

        history.append(best_fitness)

    return best_nest, best_fitness, history

# Run the optimization
best_generators, best_cost, convergence_history = cuckoo_search_opf()

# Display Results
print("--- Optimal Power Flow Optimization Results ---")
for i, p_val in enumerate(best_generators):
    print(f"Generator {i+1} Power Output: {p_val:.4f} MW")

total_generation = np.sum(best_generators)
losses = calculate_loss(best_generators)
print(f"Total Generation: {total_generation:.4f} MW")
print(f"Transmission Loss: {losses:.4f} MW")
print(f"Net Power (Generation - Loss): {total_generation - losses:.4f} MW (Target Demand: {PD} MW)")
print(f"Optimized Generation Fuel Cost: {best_cost:.4f} $/hr (including penalty adjustment)")


--- Optimal Power Flow Optimization Results ---
Generator 1 Power Output: 49.4092 MW
Generator 2 Power Output: 50.8077 MW
Generator 3 Power Output: 50.5598 MW
Total Generation: 150.7767 MW
Transmission Loss: 0.7789 MW
Net Power (Generation - Loss): 149.9979 MW (Target Demand: 150.0 MW)
Optimized Generation Fuel Cost: 657.5667 $/hr (including penalty adjustment)
